In [3]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 16000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 2000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 2000
})


In [2]:
from datasets import load_dataset, Dataset

dataset = load_dataset("dair-ai/emotion")

train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 16000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 2000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 2000
})


In [4]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully.")

Tokenizer loaded successfully.


In [5]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

print(tokenized_train)
print(tokenized_val)
print(tokenized_test)

Map: 100%|██████████| 2000/2000 [00:00<00:00, 14668.50 examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 16000
})
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})


In [6]:
from transformers import AutoModelForSequenceClassification

label2id = {
    "sadness": 0,
    "joy": 1,
    "love": 2,
    "anger": 3,
    "fear": 4,
    "surprise": 5
}

id2label = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=6,
    id2label=id2label,
    label2id=label2id
)

print("BERT emotion model loaded successfully.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3465.10it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

BERT emotion model loaded successfully.


In [8]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    return {
        "accuracy": accuracy,
        "f1": macro_f1
    }

In [9]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_emotion_results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

print("Training arguments configured successfully.")

Training arguments configured successfully.


In [10]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Trainer created successfully.")

Trainer created successfully.


In [13]:
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ]
)

print("Trainer with Early Stopping created successfully.")

Trainer with Early Stopping created successfully.


In [14]:
train_output = trainer.train()

print(train_output)

Step,Training Loss,Validation Loss,Accuracy,F1
500,0.453204,0.293442,0.931500,0.906465
1000,0.457648,0.290965,0.932500,0.911077
1500,0.292397,0.312919,0.937500,0.913804
2000,0.280000,0.344127,0.935500,0.909267
2500,0.302597,0.328713,0.939000,0.912490


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.39it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

TrainOutput(global_step=2500, training_loss=0.3916651565551758, metrics={'train_runtime': 473.0944, 'train_samples_per_second': 101.46, 'train_steps_per_second': 6.341, 'total_flos': 2631205048320000.0, 'train_loss': 0.3916651565551758, 'epoch': 2.5})


In [15]:
test_results = trainer.evaluate(
    tokenized_test
)

print(test_results)

Training Loss,Validation Loss,Step,Accuracy,F1
0.302597,0.337725,2500,0.925000,0.881739


{'eval_loss': 0.3377252221107483, 'eval_accuracy': 0.925, 'eval_f1': 0.8817389316326896}


In [16]:
from sklearn.metrics import classification_report

predictions = trainer.predict(tokenized_test)

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=-1)

print(classification_report(
    y_true,
    y_pred,
    target_names=[
        "sadness",
        "joy",
        "love",
        "anger",
        "fear",
        "surprise"
    ]
))

              precision    recall  f1-score   support

     sadness       0.97      0.96      0.97       581
         joy       0.94      0.95      0.95       695
        love       0.81      0.81      0.81       159
       anger       0.94      0.90      0.92       275
        fear       0.88      0.88      0.88       224
    surprise       0.71      0.83      0.76        66

    accuracy                           0.93      2000
   macro avg       0.88      0.89      0.88      2000
weighted avg       0.93      0.93      0.93      2000



In [17]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation F1:", trainer.state.best_metric)

Best checkpoint: ./bert_emotion_results\checkpoint-1500
Best validation F1: 0.9138035297239592


In [18]:
from transformers import AutoModelForSequenceClassification

best_model = AutoModelForSequenceClassification.from_pretrained(
    trainer.state.best_model_checkpoint
)

print("Model loaded successfully.")
print("Number of labels:", best_model.config.num_labels)
print("Labels:", best_model.config.id2label)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6423.03it/s]

Model loaded successfully.
Number of labels: 6
Labels: {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}


In [19]:
import os

final_emotion_path = "../../models/emotion/bert_emotion"

os.makedirs(final_emotion_path, exist_ok=True)

best_model.save_pretrained(final_emotion_path)
tokenizer.save_pretrained(final_emotion_path)

print("Emotion model saved successfully.")
print("Path:", final_emotion_path)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.93it/s]

Emotion model saved successfully.
Path: ../../models/emotion/bert_emotion


In [20]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

saved_model = AutoModelForSequenceClassification.from_pretrained(
    "../../models/emotion/bert_emotion"
)

saved_tokenizer = AutoTokenizer.from_pretrained(
    "../../models/emotion/bert_emotion"
)

print("Saved model loaded successfully.")
print("Model:", saved_model.config._name_or_path)
print("Labels:", saved_model.config.id2label)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8892.80it/s]

Saved model loaded successfully.
Model: ../../models/emotion/bert_emotion
Labels: {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}
